<a href="https://colab.research.google.com/github/Fdiannz/BT4222-Project-Group-4--MIND/blob/main/step4_advanced_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# step4_advanced_model: NRMS (Neural News Recommendation with Multi-Head Self-Attention)

**Owner:** P5
**Purpose:** a content-based neural model that scores articles from their **title words** and the user's **click history**, so newly published (cold) articles can be ranked without any click history.
**Inputs:** `data/processed/` from step 1 (`news_metadata`, `fit_*`, `tune_*`, `eval_*`)
**Outputs:** score files `data/scores/step4_nrms_{train_sample,tune,eval}.parquet` (`config.SCORES_DIR`) (`impression_id, news_id, score`), used by step 5 (MMR) and step 6 (diagnostics); checkpoints and logs in `outputs/step4_nrms/`
**Runtime:** Colab **GPU** (*Runtime → Change runtime type → T4 GPU*). Run top to bottom; keep outputs.

**How NRMS works (Wu et al., 2019)**
1. **News encoder:** title words → pre-trained word2vec vectors → multi-head self-attention between words → additive attention pools the words into one article vector.
2. **User encoder:** the same news encoder turns each clicked history article into a vector → multi-head self-attention between history articles → additive attention pools them into one user vector.
3. **Click score** = user vector · candidate article vector.

**Where each part comes from in the course**

| NRMS component | Lecture / notebook |
|---|---|
| Pre-trained word embeddings (word2vec-google-news-300) | Topic 6 Embedding |
| Tokenise, pad, embedding lookup | Topic 7 (Sentiment Analysis LSTM example) |
| Self-attention, multi-head attention | Topic 7 Part B (Transformer) |
| Attention pooling, inspecting attention weights | Sequential Recommendation LSTM with Attention notebook |
| 1 click + 4 within-impression negatives, softmax cross-entropy on raw logits | Topic 6/7 negative sampling & BPR; ML Optimization notebook |
| Per-impression AUC / MRR / nDCG on full slates | Topic 4 Evaluation (restricted candidate set bias) |

**Design choices:** the encoder attends over title **words** (not one pre-computed vector per article), all metrics are on **full slates**, settings are chosen on the **tune split** and the test split is used once, and results are split by **cold vs warm** articles. The comparison with the past group's baselines is in the report (aspect 2) and step 2.

## 0. Setup

In [18]:
# --- Setup: run this first, every session ---
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/BT4222-Project-Group-4--MIND && git clone -q https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git /content/BT4222-Project-Group-4--MIND
import sys; sys.path.insert(0, '/content/BT4222-Project-Group-4--MIND')
!pip -q install gensim

import config
print("Data folder:", config.PROCESSED_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data folder: /content/drive/MyDrive/BT4222_Project/data/processed


## 1. Settings

All experiment settings live in `CFG`. `COLS` maps our standard column names to the names in step 1's parquet files. If section 2 reports a missing column, fix the name here, not further down.

In [19]:
import os, re, json, time, math
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from utils.metrics import evaluate, evaluate_by_segment, attach_scores

PROC = config.PROCESSED_DIR
OUTPUT_DIR = getattr(config, "OUTPUT_DIR", os.path.join(os.path.dirname(os.path.dirname(PROC)), "outputs"))
RUN_DIR = os.path.join(OUTPUT_DIR, "step4_nrms")      # checkpoints, logs, cached word vectors
SCORE_DIR = getattr(config, "SCORES_DIR", os.path.join(OUTPUT_DIR, "scores"))   # shared score files (config.SCORES_DIR)
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(SCORE_DIR, exist_ok=True)

FILES = {
    "news": "news_metadata.parquet",
    "fit_imp": "fit_impressions.parquet",   "fit_cand": "fit_candidates_sampled.parquet",
    "tune_imp": "tune_impressions.parquet", "tune_cand": "tune_candidates.parquet",
    "eval_imp": "eval_impressions.parquet", "eval_cand": "eval_candidates.parquet",
}

# standard name -> name in the parquet files (edit the right-hand side if step 1 used other names)
COLS = {
    "impression_id": "impression_id",
    "news_id": "news_id",
    "label": "label",
    "is_cold": "cold_article",       # step 1: 1 if prior_exposure_count < config.COLD_THRESHOLD
    "history": "history",
    "impressions": "impressions",   # full slate string in *_impressions (only needed for train-slate metrics)
    "title": "title",
    "abstract": "abstract",
    "category": "category",
}

CFG = dict(
    text_fields=("title",), field_lens=(30,),  # words kept per field
    max_hist=50,             # history articles per user (MIND standard)
    neg_k=4,                 # negatives per click (1:4, as in step 1)
    d_model=256, n_heads=16, query_dim=200, dropout=0.2,
    lr=1e-4, batch_size=64, epochs=6,
    n_train_clicks=1_500_000, # training subset (MIND-large is too big for Colab per epoch); stated in the report
    n_tune_imps=20_000,      # tune impressions scored after each epoch (model selection)
    n_train_slate_imps=20_000,  # fit impressions scored on full slates for the train-vs-test comparison
    w2v_limit=500_000,       # most frequent word2vec words loaded (saves RAM)
    seed=getattr(config, "SEED", 42),
)
USE_W2V = True      # False = random word vectors (only for a quick dry run)
RESUME = True       # continue from the last saved epoch if Colab disconnected
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"   # mixed precision: faster on GPU

def seed_everything(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])
print("Device:", DEVICE)

Device: cuda


## 2. Load step 1 data and check columns

Only the needed columns are loaded. Columns are renamed to the standard names in `COLS`, so the rest of the notebook (and `utils/metrics.py`) can rely on `impression_id`, `news_id`, `label`, `is_cold`.

In [20]:
def load(name, want):
    path = os.path.join(PROC, FILES[name])
    have = pq.read_schema(path).names
    cols = [COLS[c] for c in want if COLS[c] in have]
    df = pd.read_parquet(path, columns=cols).rename(columns={COLS[c]: c for c in want})
    print(f"{name:10s} {len(df):>12,} rows | file columns: {have}")
    return df, have

NEED = {
    "news": ["news_id", "title"], "fit_cand": ["impression_id", "news_id", "label"],
    "fit_imp": ["impression_id", "history"],
    "tune_cand": ["impression_id", "news_id", "label", "is_cold"], "tune_imp": ["impression_id", "history"],
    "eval_cand": ["impression_id", "news_id", "label", "is_cold"], "eval_imp": ["impression_id", "history"],
}
OPTIONAL = {"news": ["abstract", "category"],
            "tune_cand": ["category"], "eval_cand": ["category"]}

data, schemas = {}, {}
for name in FILES:
    data[name], schemas[name] = load(name, NEED[name] + OPTIONAL.get(name, []))

missing = {n: [c for c in NEED[n] if c not in data[n].columns] for n in NEED}
missing = {n: c for n, c in missing.items() if c}
assert not missing, f"Missing columns {missing}: update COLS in section 1 to match the file columns printed above."
print("All required columns found.")

news, fit_cand, fit_imp = data["news"], data["fit_cand"], data["fit_imp"]
tune_cand, tune_imp, eval_cand, eval_imp = data["tune_cand"], data["tune_imp"], data["eval_cand"], data["eval_imp"]
for name in ["fit_cand", "tune_cand", "eval_cand"]:
    d = data[name]
    print(f"{name}: {d.impression_id.nunique():,} impressions, click rate {d.label.mean():.3f}"
          + (f", cold share {d.is_cold.mean():.3f}" if "is_cold" in d else ""))

news            104,151 rows | file columns: ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities', 'news_idx']
fit_imp       1,801,231 rows | file columns: ['impression_id', 'user_id', 'time', 'history', 'impressions', 'user_idx']
fit_cand     12,894,210 rows | file columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']
tune_imp        431,517 rows | file columns: ['impression_id', 'user_id', 'time', 'history', 'impressions', 'user_idx']
tune_cand    17,400,106 rows | file columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']
eval_imp        376,471 rows | file columns: ['impression_id', 'user_id', 'time', 'history', 'impressions', 'user_idx']
eval_cand    14,085,557 rows | file columns: ['impression_id', 'user_id', 'user_idx', 'time

## 3. Tokenise article text

Each article becomes a fixed-length array of word IDs (lower-cased, padded with 0), like the tokenise-and-pad step in the Topic 7 LSTM example. Row 0 is a padding article used to fill short histories.

The vocabulary is built from article text only (no clicks or labels), so it does not leak evaluation outcomes.

In [21]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower()) if isinstance(text, str) else []

news = news.drop_duplicates("news_id").reset_index(drop=True)
NEWS_ROW = pd.Series(np.arange(1, len(news) + 1), index=news["news_id"].to_numpy())  # news_id -> row (0 = padding)

def build_vocab(fields):
    counts = {}
    for f in fields:
        for t in news[f].fillna("") if f in news else []:
            for w in tokenize(t):
                counts[w] = counts.get(w, 0) + 1
    words = sorted(w for w, c in counts.items() if c >= 2)        # rare words -> <unk>
    vocab = {"<pad>": 0, "<unk>": 1}
    vocab.update({w: i + 2 for i, w in enumerate(words)})
    return vocab

def build_tokens(fields, lens, vocab):
    tok = np.zeros((len(news) + 1, sum(lens)), dtype=np.int64)
    start = 0
    for f, L in zip(fields, lens):
        col = news[f].fillna("") if f in news else pd.Series([""] * len(news))
        for i, text in enumerate(col):
            ids = [vocab.get(w, 1) for w in tokenize(text)][:L]
            tok[i + 1, start:start + len(ids)] = ids
        start += L
    return tok

VOCAB = build_vocab(("title", "abstract") if "abstract" in news else ("title",))
TOKENS = build_tokens(CFG["text_fields"], CFG["field_lens"], VOCAB)
title_len = (TOKENS[1:, :CFG["field_lens"][0]] > 0).sum(1)
print(f"Vocabulary: {len(VOCAB):,} words | articles: {len(news):,} | "
      f"title words kept: median {np.median(title_len):.0f}, empty titles {np.mean(title_len == 0):.2%}")
print("Example:", news.loc[0, "title"], "->", TOKENS[1, :12])

Vocabulary: 53,058 words | articles: 104,151 | title words kept: median 11, empty titles 0.00%
Example: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By -> [47395  7206 38098 16142 37334  9461  3274 37334 35954 46352  8127     0]


## 4. Word vectors: word2vec-google-news-300 (Topic 6)

Each vocabulary word starts from its pre-trained word2vec vector, which is trained on Google News, the same domain as MIND. Words not in word2vec (including stop words like *a*, *of*) start from small random vectors. All vectors are fine-tuned during training.

The download (about 1.6 GB) happens once. The resulting matrix is cached on Drive, so later sessions skip it.

In [22]:
EMB_DIM = 300
def build_word_vectors(vocab):
    cache = os.path.join(RUN_DIR, f"word2vec_init_{len(vocab)}.npy")
    if os.path.exists(cache):
        print("Loaded cached word vectors:", cache)
        return np.load(cache)
    rng = np.random.default_rng(CFG["seed"])
    emb = rng.normal(0, 0.1, (len(vocab), EMB_DIM)).astype(np.float32)
    emb[0] = 0.0
    if not USE_W2V:
        print("USE_W2V=False: random word vectors (dry run only)")
        return emb
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    path = api.load("word2vec-google-news-300", return_path=True)
    kv = KeyedVectors.load_word2vec_format(path, binary=True, limit=CFG["w2v_limit"])
    hits = 0
    for w, i in vocab.items():
        if i < 2:
            continue
        for form in (w, w.capitalize(), w.upper()):   # word2vec is case-sensitive
            if form in kv.key_to_index:
                emb[i] = kv[form]; hits += 1
                break
    print(f"word2vec coverage: {hits:,}/{len(vocab) - 2:,} words ({hits / (len(vocab) - 2):.1%})")
    np.save(cache, emb)
    del kv
    return emb

WORD_EMB = build_word_vectors(VOCAB)

Loaded cached word vectors: /content/drive/MyDrive/BT4222_Project/outputs/step4_nrms/word2vec_init_53058.npy


## 5. User histories

Each impression's click history becomes up to `max_hist` article rows (most recent kept), padded with 0. MIND gives the same history for every candidate in an impression, so it is stored once per impression.

In [23]:
def parse_history(h):
    if h is None or (isinstance(h, float) and np.isnan(h)):
        return []
    if isinstance(h, str):
        return h.split()
    return list(h)

def build_history(imp_df, impression_ids=None):
    # Returns (matrix [n_imp, max_hist] of article rows, Series impression_id -> matrix row).
    df = imp_df if impression_ids is None else imp_df[imp_df.impression_id.isin(impression_ids)]
    df = df.drop_duplicates("impression_id")
    H = np.zeros((len(df), CFG["max_hist"]), dtype=np.int64)
    for i, h in enumerate(df["history"].to_numpy()):
        ids = parse_history(h)[-CFG["max_hist"]:]
        rows = NEWS_ROW.reindex(ids).fillna(0).to_numpy(dtype=np.int64) if ids else []
        H[i, :len(rows)] = rows
    index = pd.Series(np.arange(len(df)), index=df["impression_id"].to_numpy())
    return H, index

tune_H, tune_idx = build_history(tune_imp)
eval_H, eval_idx = build_history(eval_imp)
for name, H in [("tune", tune_H), ("eval", eval_H)]:
    n = (H > 0).sum(1)
    print(f"{name}: {len(H):,} impressions | history length median {np.median(n):.0f}, empty {np.mean(n == 0):.2%}")

tune: 431,517 impressions | history length median 20, empty 2.64%
eval: 376,471 impressions | history length median 19, empty 2.99%


## 6. Training samples: 1 click + 4 negatives from the same impression

From `fit_candidates_sampled`, each click becomes one sample with 4 unclicked articles **from the same impression**, so every negative was actually shown to the user (interim Section 3.3). A random subset of `n_train_clicks` clicks is used to fit Colab's time limits. This sampling choice is reported in the final report.

The click is always placed first (target index 0). This is safe because the model scores each candidate independently and never sees its position.

In [24]:
def build_train_samples(cand, n_clicks, k, seed):
    rng = np.random.default_rng(seed)
    cand = cand.assign(row=NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64))
    pos = cand[cand.label == 1]
    if n_clicks and n_clicks < len(pos):
        pos = pos.sample(n=n_clicks, random_state=seed)
    neg = cand[(cand.label == 0) & cand.impression_id.isin(pos.impression_id)]
    neg = neg.assign(r=rng.random(len(neg))).sort_values(["impression_id", "r"])   # shuffle within impression
    imp_ids, start, count = np.unique(neg.impression_id.to_numpy(), return_index=True, return_counts=True)
    where = pd.Series(np.arange(len(imp_ids)), index=imp_ids)
    pos = pos[pos.impression_id.isin(imp_ids)]                                     # drop slates with no negative
    g = where.reindex(pos.impression_id.to_numpy()).to_numpy()
    j = pos.groupby("impression_id").cumcount().to_numpy()                         # j-th click in its impression
    offs = (j[:, None] * k + np.arange(k)[None, :]) % count[g][:, None]            # distinct negatives per click when possible
    neg_rows = neg["row"].to_numpy()[start[g][:, None] + offs]
    C = np.concatenate([pos["row"].to_numpy()[:, None], neg_rows], axis=1)        # [n, 1 + k], click first
    return pos["impression_id"].to_numpy(), C

train_imp_ids, train_C = build_train_samples(fit_cand, CFG["n_train_clicks"], CFG["neg_k"], CFG["seed"])
fit_H, fit_idx = build_history(fit_imp, impression_ids=np.unique(train_imp_ids))
train_H_rows = fit_idx.reindex(train_imp_ids).to_numpy()
assert not np.isnan(train_H_rows.astype(float)).any(), "some training impressions have no history row in fit_impressions"
print(f"Training samples: {len(train_C):,} clicks x {train_C.shape[1]} candidates "
      f"from {len(np.unique(train_imp_ids)):,} impressions")

Training samples: 1,500,000 clicks x 5 candidates from 1,146,261 impressions


## 7. Model

- `AdditiveAttention`: scores each item with a small learned query and returns the weighted average plus the weights, so attention can be inspected later.
- `NewsEncoder`: word vectors → projection → multi-head self-attention over words → additive attention over words.
- `UserEncoder`: multi-head self-attention over the history articles → additive attention. Users with no history get a learned "empty user" vector (about 2% of impressions).
- `NRMS.forward`: encodes each distinct article in the batch only once, then returns **raw** dot-product scores. `F.cross_entropy` applies the softmax itself (ML Optimization notebook).

In [25]:
class AdditiveAttention(nn.Module):
    def __init__(self, dim, query_dim):
        super().__init__()
        self.proj = nn.Linear(dim, query_dim)
        self.query = nn.Linear(query_dim, 1, bias=False)

    def forward(self, x, mask):                       # x [N, L, d], mask [N, L] True = real item
        a = self.query(torch.tanh(self.proj(x))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)                # -1e4 is safe in float16
        w = torch.softmax(a, dim=-1)
        return (w.unsqueeze(-1) * x).sum(1), w


def _safe_mask(mask):
    # Attention cannot run over a row with no real items, so unmask the first slot of empty rows.
    empty = ~mask.any(1)
    mask = mask.clone()
    mask[empty, 0] = True
    return mask, empty


class NewsEncoder(nn.Module):
    def __init__(self, word_emb, d_model, n_heads, query_dim, dropout):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.as_tensor(word_emb), freeze=False, padding_idx=0)
        self.proj = nn.Linear(word_emb.shape[1], d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, tokens, return_weights=False):  # tokens [N, L]
        mask, _ = _safe_mask(tokens > 0)
        x = self.proj(self.drop(self.emb(tokens)))
        h, _ = self.mha(x, x, x, key_padding_mask=~mask, need_weights=False)
        v, w = self.att(self.drop(h), mask)
        return (v, w) if return_weights else v


class UserEncoder(nn.Module):
    def __init__(self, d_model, n_heads, query_dim, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.att = AdditiveAttention(d_model, query_dim)
        self.drop = nn.Dropout(dropout)
        self.empty_user = nn.Parameter(torch.zeros(d_model))

    def forward(self, hist_vecs, hist_mask, return_weights=False):   # [B, H, d], [B, H]
        mask, empty = _safe_mask(hist_mask)
        h, _ = self.mha(hist_vecs, hist_vecs, hist_vecs, key_padding_mask=~mask, need_weights=False)
        u, w = self.att(self.drop(h), mask)
        u = torch.where(empty.unsqueeze(1), self.empty_user.to(u.dtype).expand_as(u), u)
        return (u, w) if return_weights else u


class NRMS(nn.Module):
    def __init__(self, word_emb, tokens, cfg):
        super().__init__()
        self.register_buffer("tokens", torch.as_tensor(tokens), persistent=False)
        self.news_encoder = NewsEncoder(word_emb, cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"])
        self.user_encoder = UserEncoder(cfg["d_model"], cfg["n_heads"], cfg["query_dim"], cfg["dropout"])

    def encode_news(self, rows):
        return self.news_encoder(self.tokens[rows])

    def forward(self, hist_rows, cand_rows):         # [B, H], [B, C] article rows
        B, H = hist_rows.shape
        uniq, inv = torch.unique(torch.cat([hist_rows.reshape(-1), cand_rows.reshape(-1)]), return_inverse=True)
        vecs = self.encode_news(uniq)
        hist_vecs = vecs[inv[:B * H]].view(B, H, -1)
        cand_vecs = vecs[inv[B * H:]].view(B, cand_rows.shape[1], -1)
        user = self.user_encoder(hist_vecs, hist_rows > 0)
        return (cand_vecs * user.unsqueeze(1)).sum(-1)  # raw scores [B, C]

_m = NRMS(WORD_EMB, TOKENS, CFG)
print(f"Parameters: {sum(p.numel() for p in _m.parameters()):,} "
      f"(word vectors {_m.news_encoder.emb.weight.numel():,})")
del _m

Parameters: 16,624,248 (word vectors 15,917,400)


## 8. Scoring full slates

To score a split, every article is encoded once, then each impression's user vector once, then each candidate is scored by a dot product. All candidates on the full slates are scored. The output matches `utils/metrics.attach_scores`: `impression_id, news_id, score`.

In [26]:
@torch.no_grad()
def encode_all_news(model, batch=2048):
    model.eval()
    out = []
    for s in range(0, model.tokens.shape[0], batch):
        rows = torch.arange(s, min(s + batch, model.tokens.shape[0]), device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            out.append(model.encode_news(rows).float())
    return torch.cat(out)

@torch.no_grad()
def score_candidates(model, cand, H, h_index, batch=1024):
    model.eval()
    news_vecs = encode_all_news(model)
    imp_codes, imp_ids = pd.factorize(cand["impression_id"])
    imp_codes = np.array(imp_codes, dtype=np.int64)        # writable copy for torch
    h_rows = h_index.reindex(imp_ids).to_numpy()
    if np.isnan(h_rows.astype(float)).any():
        raise ValueError("some candidate impressions have no history row")
    h_rows = h_rows.astype(np.int64)
    users = []
    for s in range(0, len(imp_ids), batch):
        hist = torch.as_tensor(H[h_rows[s:s + batch]], device=DEVICE)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            users.append(model.user_encoder(news_vecs[hist], hist > 0).float())
    users = torch.cat(users)
    news_rows = NEWS_ROW.reindex(cand["news_id"].to_numpy()).fillna(0).to_numpy(dtype=np.int64)
    if (news_rows == 0).any():
        print(f"warning: {(news_rows == 0).sum():,} candidates missing from news_metadata")
    scores = np.empty(len(cand), dtype=np.float32)
    for s in range(0, len(cand), 200_000):
        u = users[torch.as_tensor(imp_codes[s:s + 200_000], device=DEVICE)]
        v = news_vecs[torch.as_tensor(news_rows[s:s + 200_000], device=DEVICE)]
        scores[s:s + 200_000] = (u * v).sum(-1).cpu().numpy()
    return pd.DataFrame({"impression_id": cand["impression_id"].to_numpy(),
                         "news_id": cand["news_id"].to_numpy(), "score": scores})

def sample_impressions(cand, n, seed):
    ids = cand["impression_id"].drop_duplicates()
    ids = ids.sample(n=min(n, len(ids)), random_state=seed)
    return cand[cand["impression_id"].isin(ids)]

tune_small = sample_impressions(tune_cand, CFG["n_tune_imps"], CFG["seed"])   # fixed subset for epoch-by-epoch selection
print(f"Tune subset for model selection: {tune_small.impression_id.nunique():,} impressions")

Tune subset for model selection: 20,000 impressions


## 9. Training

- Loss: softmax cross-entropy over [click, 4 negatives] with target 0, computed on raw scores.
- Optimiser: Adam with learning rate 1e-4, mini-batches of 64.
- After each epoch, the model is scored on the fixed tune subset (14 Nov, full slates). The epoch with the best **tune AUC** is kept; training scores are never used for model selection.
- A checkpoint is saved to Drive after every epoch. With `RESUME = True`, a disconnected run continues where it stopped.

In [27]:
def train_nrms(cfg, run_name, train_H, train_H_rows, train_C, tune=None, log_every=500):
    tune_c, tune_h, tune_i = tune if tune is not None else (tune_small, tune_H, tune_idx)
    seed_everything(cfg["seed"])
    model = NRMS(WORD_EMB, TOKENS, cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    last_ckpt = os.path.join(RUN_DIR, f"{run_name}_last.pt")
    best_ckpt = os.path.join(RUN_DIR, f"{run_name}_best.pt")
    log, start_epoch, best_auc = [], 0, -1.0
    if RESUME and os.path.exists(last_ckpt):
        st = torch.load(last_ckpt, map_location=DEVICE)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        log, start_epoch, best_auc = st["log"], st["epoch"] + 1, st["best_auc"]
        print(f"Resumed {run_name} after epoch {start_epoch}")

    H_dev = torch.as_tensor(train_H, device=DEVICE)
    rows_dev = torch.as_tensor(train_H_rows.astype(np.int64), device=DEVICE)
    C_dev = torch.as_tensor(train_C, device=DEVICE)
    n, bs = len(C_dev), cfg["batch_size"]
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        t0, total, perm = time.time(), 0.0, torch.randperm(n, device=DEVICE)
        for b, s in enumerate(range(0, n, bs)):
            idx = perm[s:s + bs]
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                logits = model(H_dev[rows_dev[idx]], C_dev[idx])
                loss = F.cross_entropy(logits.float(), torch.zeros(len(idx), dtype=torch.long, device=DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item() * len(idx)
            if log_every and b % log_every == 0:
                print(f"  epoch {epoch + 1} batch {b:>6}/{math.ceil(n / bs)} loss {loss.item():.4f}")
        tune_scores = score_candidates(model, tune_c, tune_h, tune_i)
        tune_res = evaluate(attach_scores(tune_c, tune_scores), "score")
        row = {"epoch": epoch + 1, "train_loss": total / n, **{f"tune_{k}": v for k, v in tune_res.items()},
               "minutes": (time.time() - t0) / 60}
        log.append(row)
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
        if tune_res["auc"] > best_auc:
            best_auc = tune_res["auc"]
            torch.save({"model": model.state_dict(), "cfg": cfg, "epoch": epoch + 1}, best_ckpt)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch,
                    "log": log, "best_auc": best_auc}, last_ckpt)

    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE)["model"])
    log = pd.DataFrame(log)
    log.to_csv(os.path.join(RUN_DIR, f"{run_name}_log.csv"), index=False)
    best = log.loc[log["tune_auc"].idxmax()]
    print(f"{run_name}: best epoch {int(best.epoch)} (tune AUC {best.tune_auc:.4f})")
    return model, log

## 10. Train the main model

In [28]:
RUN = "nrms_main_v2"
model, train_log = train_nrms(CFG, RUN, fit_H, train_H_rows, train_C)
with open(os.path.join(RUN_DIR, f"{RUN}_config.json"), "w") as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()}, f, indent=2)
train_log

  epoch 1 batch      0/23438 loss 1.6095
  epoch 1 batch    500/23438 loss 1.4665
  epoch 1 batch   1000/23438 loss 1.4080
  epoch 1 batch   1500/23438 loss 1.4995
  epoch 1 batch   2000/23438 loss 1.4085
  epoch 1 batch   2500/23438 loss 1.4461
  epoch 1 batch   3000/23438 loss 1.2909
  epoch 1 batch   3500/23438 loss 1.3539
  epoch 1 batch   4000/23438 loss 1.4020
  epoch 1 batch   4500/23438 loss 1.3168
  epoch 1 batch   5000/23438 loss 1.4968
  epoch 1 batch   5500/23438 loss 1.3018
  epoch 1 batch   6000/23438 loss 1.4252
  epoch 1 batch   6500/23438 loss 1.2747
  epoch 1 batch   7000/23438 loss 1.3842
  epoch 1 batch   7500/23438 loss 1.4276
  epoch 1 batch   8000/23438 loss 1.4042
  epoch 1 batch   8500/23438 loss 1.4203
  epoch 1 batch   9000/23438 loss 1.3191
  epoch 1 batch   9500/23438 loss 1.3926
  epoch 1 batch  10000/23438 loss 1.3442
  epoch 1 batch  10500/23438 loss 1.2744
  epoch 1 batch  11000/23438 loss 1.2431
  epoch 1 batch  11500/23438 loss 1.2031
  epoch 1 batch 

,epoch,train_loss,tune_auc,tune_mrr,tune_ndcg@5,tune_ndcg@10,tune_n_impressions,tune_n_skipped,minutes
0,1,1.349722,0.6515,0.2905,0.3119,0.3739,20000,0,16.386301
1,2,1.286087,0.6570,0.2936,0.3165,0.3787,20000,0,16.396993
2,3,1.263527,0.6678,0.3060,0.3292,0.3920,20000,0,16.440832
3,4,1.250235,0.6725,0.3072,0.3315,0.3930,20000,0,16.484929
4,5,1.239052,0.6749,0.3116,0.3365,0.3984,20000,0,16.461008
5,6,1.229553,0.6750,0.3120,0.3372,0.3985,20000,0,16.424089


## 11. 📊 Results: train vs tune vs test, overall and cold vs warm

All three are measured the same way: full slates, per-impression metrics from `utils/metrics.py`.
- **Train:** a sample of fit impressions (9–13 Nov) with all their displayed candidates. This needs the full slates in `fit_impressions`; if that column is missing, ask P1.
- **Tune:** 14 Nov (used for model selection).
- **Test:** the official dev split, 15 Nov, scored once with the selected model.

Score files are saved to `config.SCORES_DIR` (`data/scores/`) for step 5 (MMR) and step 6 (diagnostics).

In [29]:
def full_slates_from_impressions(imp_df, n, seed):
    # Expand the 'N123-1 N456-0 ...' slate strings of a sample of fit impressions into candidate rows.
    sample = imp_df.drop_duplicates("impression_id").sample(n=min(n, imp_df.impression_id.nunique()), random_state=seed)
    rows = sample[["impression_id", "impressions"]].assign(cand=sample["impressions"].str.split()).explode("cand")
    nid_lab = rows["cand"].str.rsplit("-", n=1)
    out = pd.DataFrame({"impression_id": rows["impression_id"].to_numpy(),
                        "news_id": nid_lab.str[0].to_numpy(),
                        "label": nid_lab.str[1].astype(int).to_numpy()})
    if pd.api.types.is_integer_dtype(news["news_id"]):           # step 1 may have mapped IDs to integers
        ids = out["news_id"].str.lstrip("N")
        if not ids.str.isdigit().all():
            raise ValueError("slate IDs do not match the integer news_id in news_metadata: ask P1 for the mapping")
        out["news_id"] = ids.astype(news["news_id"].dtype)
    return out

results, scored = {}, {}
if COLS["impressions"] in schemas["fit_imp"]:
    slates = pd.read_parquet(os.path.join(PROC, FILES["fit_imp"]),
                             columns=[COLS["impression_id"], COLS["impressions"]])
    slates.columns = ["impression_id", "impressions"]
    tr = full_slates_from_impressions(slates, CFG["n_train_slate_imps"], CFG["seed"])
    del slates
    tr_H, tr_idx = build_history(fit_imp, impression_ids=tr.impression_id.unique())
    s = score_candidates(model, tr, tr_H, tr_idx)
    s.to_parquet(os.path.join(SCORE_DIR, "step4_nrms_train_sample.parquet"), index=False)
    results["train (full slates)"] = evaluate(attach_scores(tr, s), "score")
else:
    print("fit_impressions has no full-slate column: train metrics on full slates skipped (ask P1).")

for split, cand, H, idx in [("tune", tune_cand, tune_H, tune_idx), ("eval", eval_cand, eval_H, eval_idx)]:
    s = score_candidates(model, cand, H, idx)
    s.to_parquet(os.path.join(SCORE_DIR, f"step4_nrms_{split}.parquet"), index=False)
    scored[split] = attach_scores(cand, s)
    print(f"saved {split} scores: {len(s):,} candidates")

for split in ["tune", "eval"]:
    results[split] = evaluate(scored[split], "score")
overall = pd.DataFrame(results).T
print("Overall (full slates):")
display(overall)

print("Test (eval, 15 Nov) by impression segment: cold = every clicked article was cold:")
by_segment = evaluate_by_segment(scored["eval"], "score", "is_cold")
display(by_segment)
by_segment.to_csv(os.path.join(RUN_DIR, f"{RUN}_eval_by_segment.csv"))

saved tune scores: 17,400,106 candidates
saved eval scores: 14,085,557 candidates
Overall (full slates):


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
train (full slates),0.7662,0.4190,0.4620,0.5148,20000.0,0.0
tune,0.6757,0.3109,0.3365,0.3984,431517.0,0.0
eval,0.6288,0.2907,0.3162,0.3829,376471.0,0.0


Test (eval, 15 Nov) by impression segment: cold = every clicked article was cold:


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
all,0.6288,0.2907,0.3162,0.3829,376471,0
cold,0.5767,0.3500,0.3847,0.4426,77199,0
warm,0.6411,0.2897,0.3095,0.3777,262807,0
mixed,0.6503,0.1729,0.2196,0.2938,36465,0


### Interpretation

_Interpretation (fill in after running): e.g. how large the train-test gap is on equal terms, and whether cold-click impressions score close to warm ones._

## 12. Ablations (architecture, aspect 3)

Each variant changes one setting relative to the main model. To fit the time budget, ablations use a smaller training subset and fewer epochs, and are compared on the **tune split only**. The test split is used once, for the selected model in section 11.

In [30]:
RUN_ABLATIONS = True
ABL_BASE = dict(CFG, n_train_clicks=200_000, epochs=2)
ABLATIONS = {
    "base (title, 16 heads, hist 50)": {},
    "history 10":                      {"max_hist": 10},
    "4 heads":                         {"n_heads": 4},
    "title + abstract":                {"text_fields": ("title", "abstract"), "field_lens": (30, 50)},
}

def run_ablation(name, overrides):
    global TOKENS, CFG
    cfg = dict(ABL_BASE, **overrides)
    saved = (TOKENS, CFG)
    try:
        CFG = cfg   # build_history and the model read the history length from CFG
        if cfg["text_fields"] != saved[1]["text_fields"]:
            TOKENS = build_tokens(cfg["text_fields"], cfg["field_lens"], VOCAB)
        imp_ids, C = build_train_samples(fit_cand, cfg["n_train_clicks"], cfg["neg_k"], cfg["seed"])
        H, idx = build_history(fit_imp, impression_ids=np.unique(imp_ids))
        t_H, t_idx = build_history(tune_imp)
        run = "abl_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        m, log = train_nrms(cfg, run, H, idx.reindex(imp_ids).to_numpy(), C,
                            tune=(tune_small, t_H, t_idx), log_every=0)
        res = evaluate(attach_scores(tune_small, score_candidates(m, tune_small, t_H, t_idx)), "score")
        return {"best_epoch": int(log.loc[log.tune_auc.idxmax(), "epoch"]), **res}
    finally:
        TOKENS, CFG = saved

if RUN_ABLATIONS:
    ablation_table = pd.DataFrame({name: run_ablation(name, o) for name, o in ABLATIONS.items()}).T
    ablation_table.to_csv(os.path.join(RUN_DIR, "ablation_tune.csv"))
    display(ablation_table)

Resumed abl_base_title_16_heads_hist_50 after epoch 2
abl_base_title_16_heads_hist_50: best epoch 2 (tune AUC 0.6238)
Resumed abl_history_10 after epoch 2
abl_history_10: best epoch 2 (tune AUC 0.6202)
Resumed abl_4_heads after epoch 2
abl_4_heads: best epoch 2 (tune AUC 0.6286)
Resumed abl_title_abstract after epoch 2
abl_title_abstract: best epoch 2 (tune AUC 0.6221)


,best_epoch,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
"base (title, 16 heads, hist 50)",2.0,0.6238,0.2646,0.2806,0.3445,20000.0,0.0
history 10,2.0,0.6202,0.2671,0.2839,0.3474,20000.0,0.0
4 heads,2.0,0.6286,0.2706,0.2866,0.3516,20000.0,0.0
title + abstract,2.0,0.6221,0.2658,0.2810,0.3461,20000.0,0.0


## 13. Inside the model: attention weights (aspect 4)

For a few test impressions:
- **Word attention:** which title words the news encoder weighted most for the clicked article.
- **History attention:** which past clicks the user encoder weighted most.

Caveat (cf. KGIN, Topic 9): attention weights show what the model relied on to make its prediction, not verified reasons why the user clicked.

In [31]:
import gc
# Free the large training tables first: section 13 only needs the model, news and the eval split.
for _name in ["fit_cand", "fit_imp", "fit_H", "train_C", "train_H_rows", "train_imp_ids", "tr", "tr_H", "data"]:
    globals().pop(_name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

INV_VOCAB = {i: w for w, i in VOCAB.items()}
NEWS_VECS = encode_all_news(model)          # encode every article once, reused for all examples

@torch.no_grad()
def explain(model, impression_id, top=5):
    model.eval()
    hist = torch.as_tensor(eval_H[[eval_idx[impression_id]]], device=DEVICE)
    _, hw = model.user_encoder(NEWS_VECS[hist], hist > 0, return_weights=True)
    hrows, hw = hist[0].cpu().numpy(), hw[0].float().cpu().numpy()
    print(f"Impression {impression_id}: most-attended history articles")
    for j in np.argsort(-hw)[:3]:
        if hrows[j] > 0:
            print(f"   {hw[j]:.2f}  {news.loc[hrows[j] - 1, 'title']}")
    clicked = eval_cand.loc[(eval_cand.impression_id == impression_id) & (eval_cand.label == 1), "news_id"]
    for nid in clicked:
        row = int(NEWS_ROW[nid])
        tok = model.tokens[[row]]
        _, ww = model.news_encoder(tok, return_weights=True)
        words = [(INV_VOCAB[int(t)], float(w)) for t, w in zip(tok[0].cpu(), ww[0].float().cpu()) if int(t) > 0]
        print(f"Clicked: {news.loc[row - 1, 'title']}")
        print("   top words:", ", ".join(f"{w} ({a:.2f})" for w, a in sorted(words, key=lambda x: -x[1])[:top]))

examples = eval_cand.loc[eval_cand.label == 1, "impression_id"].drop_duplicates().sample(3, random_state=CFG["seed"])
for imp in examples:
    explain(model, imp)
    print()

Impression 90453: most-attended history articles
   0.08  'Go to confession': Scaramucci urges Mick Mulvaney to leave the White House
   0.07  Democrats' 2020 race has a new shadow: Hillary Clinton
   0.07  Joe Biden reportedly denied Communion at a South Carolina church because of his stance on abortion
Clicked: The Unlikely Star of My Family's Thanksgiving Table
   top words: my (0.14), family's (0.13), table (0.13), of (0.13), the (0.12)

Impression 210745: most-attended history articles
   0.03  Sessions wins endorsement from key GOP senators
   0.03  Loved One Gave You Down Payment Money? You'll Need a Mortgage Gift Letter
   0.03  Exclusive: Ukraine to fire prosecutor who discussed Bidens with Giuliani - source
Clicked: This Easy Trick Will Defrost Your Car Window In Seconds
   top words: defrost (0.12), trick (0.10), seconds (0.10), your (0.10), easy (0.10)

Impression 192076: most-attended history articles
   0.09  Keanu Reeves Holds Hands with Artist Alexandra Grant on the Red

### 13b. Attention in numbers

The examples above are anecdotes. This cell measures two patterns over thousands of test impressions:
1. **History attention vs interests:** how often the user's *most-attended* history article has the same category as the article they clicked, compared with a *random* history article. If the first is clearly higher, history attention is picking up real interests.
2. **Word attention:** average weight on stopwords (*the, my, of…*) vs content words, relative to equal attention (1.0 = every word weighted the same).

In [32]:
STOPWORDS = set("the a an of to in on for and or with at by from is are was were be been this that these those it its "
                "he she they his her their my your our you we i as after says how why what when who new".split())
CATEGORY = news["category"].to_numpy()          # article row r -> CATEGORY[r - 1]

@torch.no_grad()
def attention_stats(model, n_imps=3000, seed=CFG["seed"]):
    model.eval()
    rng = np.random.default_rng(seed)
    clicks = eval_cand.loc[eval_cand.label == 1, ["impression_id", "news_id"]]
    imps = clicks["impression_id"].drop_duplicates()
    imps = imps.sample(n=min(n_imps, len(imps)), random_state=seed).to_numpy()
    clicked_cats = (clicks.assign(cat=CATEGORY[NEWS_ROW.reindex(clicks.news_id).to_numpy() - 1])
                          .groupby("impression_id")["cat"].agg(set))

    # 1. history attention: top-attended vs random history article
    top_hit, rand_hit = [], []
    for s in range(0, len(imps), 1000):
        batch = imps[s:s + 1000]
        hist = torch.as_tensor(eval_H[eval_idx.reindex(batch).to_numpy()], device=DEVICE)
        _, hw = model.user_encoder(NEWS_VECS[hist], hist > 0, return_weights=True)
        hist, hw = hist.cpu().numpy(), hw.float().cpu().numpy()
        for imp, h, w in zip(batch, hist, hw):
            real = np.flatnonzero(h > 0)
            if len(real) < 2:
                continue
            cats = clicked_cats[imp]
            top_hit.append(CATEGORY[h[real[np.argmax(w[real])]] - 1] in cats)
            rand_hit.append(CATEGORY[h[rng.choice(real)] - 1] in cats)

    # 2. word attention: stopwords vs content words (weight x number of words; 1.0 = equal attention)
    rows = NEWS_ROW.reindex(clicks.loc[clicks.impression_id.isin(imps), "news_id"].unique()).to_numpy()
    stop_w, content_w = [], []
    for s in range(0, len(rows), 2000):
        tok = model.tokens[torch.as_tensor(rows[s:s + 2000], device=DEVICE)]
        _, ww = model.news_encoder(tok, return_weights=True)
        tok, ww = tok.cpu().numpy(), ww.float().cpu().numpy()
        for t, w in zip(tok, ww):
            real = t > 0
            n = real.sum()
            for ti, wi in zip(t[real], w[real]):
                (stop_w if INV_VOCAB[int(ti)] in STOPWORDS else content_w).append(wi * n)

    return pd.DataFrame({
        "value": [np.mean(top_hit), np.mean(rand_hit), np.mean(stop_w), np.mean(content_w)],
        "n": [len(top_hit), len(rand_hit), len(stop_w), len(content_w)],
    }, index=["history: top-attended article shares clicked category",
              "history: random article shares clicked category",
              "words: relative weight on stopwords (1 = equal)",
              "words: relative weight on content words (1 = equal)"]).round(3)

attention_table = attention_stats(model)
attention_table.to_csv(os.path.join(RUN_DIR, f"{RUN}_attention_stats.csv"))
display(attention_table)

,value,n
history: top-attended article shares clicked category,0.283,2861
history: random article shares clicked category,0.257,2861
words: relative weight on stopwords (1 = equal),0.941,1669
words: relative weight on content words (1 = equal),1.018,5394


## 14. Summary

_Fill in after running:_
- Selected configuration and best epoch (tune AUC):
- Test (15 Nov) overall: AUC / MRR / nDCG@5 / nDCG@10
- Cold vs warm: does the content-based model keep cold-click impressions close to warm ones?
- Train vs test gap on full slates:
- Ablation takeaways:
- Hand-off: score files in `data/scores/step4_nrms_*.parquet` for step 5 (MMR) and step 6 (diagnostics).